# 01 · Full Fine-Tuning Basics

This notebook builds the mental model for fine-tuning by doing a **full** fine-tune of a tiny model.
It runs on CPU (slowly) or any GPU. Read [docs/01](../docs/01_what_is_finetuning.md) and [docs/02](../docs/02_llm_basics.md) first.

**What you'll do:** load a tiny pre-trained model → format data → train on next-token prediction → generate before/after.


## 1. Install dependencies
Run once. On Colab, restart the runtime afterwards if prompted.


In [ ]:
!pip install -q transformers datasets accelerate


## 2. Load a tiny base model + tokenizer
We use `sshleifer/tiny-gpt2` purely so the cell runs fast anywhere. The ideas are identical for a 7B model.


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = 'sshleifer/tiny-gpt2'  # swap for 'gpt2' or a 7B model on GPU
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_name)
print('Parameters:', sum(p.numel() for p in model.parameters()))


## 3. Generate BEFORE fine-tuning
Establish a baseline so you can see the change.


In [ ]:
def generate(prompt, max_new_tokens=40):
    inputs = tokenizer(prompt, return_tensors='pt')
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0], skip_special_tokens=True)

print(generate('The capital of France is'))


## 4. Build a tiny dataset
Full fine-tuning trains on plain text. Here we teach a fixed style with a few repeated examples.


In [ ]:
from datasets import Dataset

examples = [
    'Q: What is LoRA? A: LoRA trains small low-rank matrices instead of all weights.',
    'Q: What is QLoRA? A: QLoRA is LoRA on a 4-bit quantized base model.',
    'Q: What is PEFT? A: PEFT means parameter-efficient fine-tuning.',
] * 20  # repeat so the tiny model can memorize the pattern
ds = Dataset.from_dict({'text': examples})

def tokenize(batch):
    out = tokenizer(batch['text'], truncation=True, padding='max_length', max_length=64)
    out['labels'] = out['input_ids'].copy()  # causal LM: labels = inputs
    return out

tokenized = ds.map(tokenize, batched=True, remove_columns=['text'])


## 5. Train (full fine-tuning)
Every weight updates. Note `labels = input_ids` — the model learns to predict each next token.


In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

args = TrainingArguments(output_dir='out_full', num_train_epochs=5,
    per_device_train_batch_size=4, learning_rate=5e-4, logging_steps=10, report_to='none')
collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)
trainer = Trainer(model=model, args=args, train_dataset=tokenized, data_collator=collator)
trainer.train()


## 6. Generate AFTER fine-tuning
The model should now lean toward the taught style.


In [ ]:
print(generate('Q: What is LoRA? A:'))


## Takeaways
- Fine-tuning = continue training on your data with the same next-token objective.
- Full fine-tuning updates **all** weights — fine for tiny models, too heavy for 7B.
- Next: [02_lora_finetuning.ipynb](02_lora_finetuning.ipynb) does the same with ~0.1% of the parameters.
